# Exploratory Data Analysis

### Import Libraries

In [2]:
from dash import Dash, html, dcc, callback, Output, Input
import fiona
import folium
from folium.plugins import MarkerCluster
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import plotly.express as px
from tqdm import tqdm

ModuleNotFoundError: No module named 'fiona'

## 1. Load and Inspect the Three Transshipment Datasets

Load three CSV files provided by Global Fishing Watch:
- **Encounter events** (11,681 rows × 9 columns): potential at‑sea meetings between a fishing vessel and a transshipment‑capable vessel.
- **Loitering events** (46,569 rows × 9 columns): periods when a transshipment vessel lingers in a small area, which may indicate waiting for a fishing vessel.
- **Transshipment vessels** (1,124 rows × 7 columns): metadata (MMSI, ship name, flag, IMO, first/last AIS timestamp) for vessels known to be capable of transshipment.

In [ ]:
# CWD is the notebooks directory, as that is where Jupyter will run its kernel in.
TRANSSHIPMENT_DIR_FROM_CWD = "../data/raw/transshipment"
ENCOUNTER_EVENTS_FILENAME = "encounter-events-v20170717.csv"
LOITERING_EVENTS_FILENAME = "loitering-events-v20170717.csv"
VESSELS_FILENAME = "transshipment-vessels-v20170717.csv"

In [ ]:
transshipment_data_dir = os.path.join(os.getcwd(), TRANSSHIPMENT_DIR_FROM_CWD)
encounter_path = os.path.join(transshipment_data_dir, ENCOUNTER_EVENTS_FILENAME)
loitering_path = os.path.join(transshipment_data_dir, LOITERING_EVENTS_FILENAME)
transshipment_path = os.path.join(transshipment_data_dir, VESSELS_FILENAME)

encounter_df = pd.read_csv(encounter_path)
loitering_df = pd.read_csv(loitering_path)
transshipment_df = pd.read_csv(transshipment_path)


print("="*20 + " Encounter " + "="*20)
print("Shape: ",encounter_df.shape)
print("Example Row: ",encounter_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Loitering " + "="*20)
print("Shape: ",loitering_df.shape)
print("Example Row: ",loitering_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Transshipment " + "="*20)
print("Shape: ",transshipment_df.shape)
print("Example Row: ",transshipment_df.head(1))
print("="*20 + "=========" + "="*20)

In [ ]:
print(encounter_df.info())
print(loitering_df.info())
print(transshipment_df.info())

The encounter events and loitering events datasets do not have any missing data that needs to be considered. However, the string timestamps will need to be converted to a numeric format to actually work with them.

Because we are analyzing transshipment behavior and not the characteristics of particular vessels, the transshipment vessel data will not be used.

## 2. Plotting Global Marine Protected Areas (MPAs)

In [ ]:
MPA_GDB_PATH_FROM_CWD = "../data/raw/global_mpa/WDPA_WDOECM_Apr2026_Public_marine.gdb"

In [ ]:
mpa_gdb_path = os.path.join(os.getcwd(), MPA_GDB_PATH_FROM_CWD)

fiona.listlayers(mpa_gdb_path)

In [ ]:
mpa_point_gdf = gpd.read_file(mpa_gdb_path, layer="WDPA_WDOECM_point_Apr2026_marine")

In [ ]:
mpa_location = mpa_point_gdf[['NAME_ENG', 'geometry']]

# Extract latitude and longitude
mpa_location['geometry'] = mpa_location['geometry'].explode(ignore_index=True)
mpa_location['lat'] = mpa_location['geometry'].y.astype(float)
mpa_location['long'] = mpa_location['geometry'].x.astype(float)
mpa_location.head()

In [ ]:
mpa_map = folium.Map(location=[0, 0], zoom_start=2, tiles="CartoDB positron")

mpa_marker_cluster = MarkerCluster()

for mpa in tqdm(mpa_location.iterrows(), desc="Generating Map...", total=mpa_location.shape[0]):
    folium.Marker(
        location=[mpa[1]["lat"], mpa[1]["long"]],
        popup=mpa[1]["NAME_ENG"],
        icon=folium.Icon(color="red")
    ).add_to(mpa_marker_cluster)

mpa_marker_cluster.add_to(mpa_map)

mpa_map

## 3. Visualizing Encounters on a Map

In [ ]:
encounter_map = folium.Map(location=[0, 0], zoom_start=2, tiles="CartoDB positron")

encounter_marker_cluster = MarkerCluster()

for ship in tqdm(encounter_df.iterrows(), desc="Generating Map...", total=encounter_df.shape[0]):
  lat, long = ship[1]["mean_latitude"], ship[1]["mean_longitude"]
  ship_name = transshipment_df[transshipment_df["mmsi"] == ship[1]["transshipment_vessel_mmsi"]]["shipname"].values
  if ship_name.size > 0:
    ship_name = ship_name[0]
  else:
    ship_name = "Unknown"

  folium.Marker(
      location=[lat, long],
      popup=ship_name,
      icon=folium.Icon(color="red", icon="ship")
  ).add_to(encounter_marker_cluster)

encounter_marker_cluster.add_to(encounter_map)

encounter_map

## 4. Visualizing Loitering Events on a Map

In [ ]:
loitering_map = folium.Map(location=[0, 0], zoom_start=2, tiles="CartoDB positron")

loitering_marker_cluster = MarkerCluster(name="Loitering Events")

for event in tqdm(loitering_df.iterrows(), desc="Generating Loitering Map...", total=loitering_df.shape[0]):
    lat, long = event[1]["starting_latitude"], event[1]["starting_longitude"]
    ship_name = transshipment_df[transshipment_df["mmsi"] == event[1]["transshipment_mmsi"]]["shipname"].values
    if ship_name.size > 0:
        ship_name = ship_name[0]
    else:
        ship_name = "Unknown"

    folium.Marker(
        location=[lat, long],
        popup=folium.Popup(
            f"<b>{ship_name}</b><br>Duration: {event[1]['total_event_duration']:.1f} hrs<br>Median Speed: {event[1]['median_speed_knots']:.2f} knots",
            max_width=250,
        ),
        icon=folium.Icon(color="blue", icon="ship")
    ).add_to(loitering_marker_cluster)

loitering_marker_cluster.add_to(loitering_map)

loitering_map

There are many instances where loitering events occur around marine protected areas, such as the west coast of Africa, or Oceania. It would be appropriate to try to analyze the behavior around these events to understand why they are around marine protected areas.